# Forecast Like-Game Clustering

This notebook runs Stage 1 of the workflow from the clean joined dataset: build game profiles, create weighted clustering features, fit the hierarchical clustering tree, render a dendrogram, and produce the new-game to like-game mapping table.

In [0]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipelines.forecast_clustering import (
    build_game_profiles,
    build_weighted_feature_matrix,
    find_like_games,
    fit_hierarchical_clustering,
    load_forecast_clustering_config,
    plot_dendrogram,
    select_clustering_roles,
)

In [0]:
p=r"C:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\game_attr_cleaned.csv"
df=pd.read_csv(p)
df

## 1. Load Config And Clean Joined Data

In [0]:
CONFIG_PATH = ROOT / 'config' / 'forecast_clustering' / 'default.yml'
config, _ = load_forecast_clustering_config(CONFIG_PATH)

joined = pd.read_csv(config['input_path'], low_memory=False)
joined.shape
joined

## 2. Build One Game/Cabinet Profile Per Clustered Item

In [0]:
profiles = build_game_profiles(joined, config)
profiles[['profile_id', 'theme_key', 'game_name', 'mapping_cabinet_desc', 'release_date', 'history_months']].head(10)

## 3. Create Weighted Feature Matrix

In [0]:
features = build_weighted_feature_matrix(profiles, config)
features.shape, features.iloc[:, :10].head()

## 4. Select Target Games And Qualified Historical Candidates

In [0]:
roles = select_clustering_roles(profiles, config)
role_summary = roles['role'].value_counts().rename_axis('role').reset_index(name='profiles')
target_preview = profiles.join(roles[['role']]).query("role == 'target'")[[
    'profile_id', 'theme_key', 'game_name', 'mapping_cabinet_desc', 'release_date', 'history_months'
]]
display(role_summary)
display(target_preview)

## 5. Fit Complete-Linkage Hierarchical Clustering

In [0]:
linkage_matrix = fit_hierarchical_clustering(features, config)
linkage_matrix[:5]

## 6. Dendrogram Presentation

In [0]:
dendrogram_path = plot_dendrogram(
    profiles.join(roles[['role']]),
    linkage_matrix,
    config['dendrogram_output_path'],
    config,
)
display(Image(filename=dendrogram_path))
dendrogram_path

## 7. Like-Game Mapping Table

In [0]:
like_games = find_like_games(
    profiles=profiles,
    feature_matrix=features,
    linkage_matrix=linkage_matrix,
    roles=roles,
    config=config,
)

like_games.sort_values(['target_game_name', 'target_mapping_cabinet_desc', 'candidate_rank']).head(30)

## 8. Save Notebook Outputs

In [0]:
profiles.join(roles[['is_target', 'is_qualified_historical', 'role']]).to_csv(
    config['profiles_output_path'], index=False
)
features.to_csv(config['feature_matrix_output_path'], index=False)
like_games.to_csv(config['like_games_output_path'], index=False)

{
    'profiles': config['profiles_output_path'],
    'features': config['feature_matrix_output_path'],
    'like_games': config['like_games_output_path'],
    'dendrogram': dendrogram_path,
}